# IOAI — 2025 National Selection Duplicate Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, json, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-national-selection-duplicate-detection/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
# BERT(115MB)는 github 한계로 bohrium 공개 API 의 신선한 서명 URL 로 받는다
if not os.path.isdir('data/my_custom_bert_model'):
    d = json.load(urllib.request.urlopen('https://www.bohrium.com/bohrapi/v1/square/competition/7927749324'))
    uri = [x for x in d['data']['datasetList'] if x['dsName'].endswith('train')][0]['downloadUri']
    urllib.request.urlretrieve(uri, 'train.zip')
    with zipfile.ZipFile('train.zip') as z:
        for n in z.namelist():
            if n.startswith('my_custom_bert_model/'): z.extract(n, 'data')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Duplicate Detection — 모범답안 (BERT 파인튜닝)

**데이터 준비가 핵심**: 400개 양성쌍(label 1) + raw 문장을 무작위로 짝지어 만든 음성쌍(label 0)을 합쳐
균형 학습셋을 만든다. 제공된 BERT 에 분류 헤드를 얹어(`AutoModelForSequenceClassification`) 쌍 분류로
파인튜닝하고 test 를 예측한다. GPU 수 분.

In [ ]:
import pandas as pd, json
train = pd.read_csv("data/train_data.csv")       # 동등쌍(400), is_duplicate=1
raw   = pd.read_csv("data/raw_questions.csv")     # 무관 문장 1000
test  = pd.read_csv("data/test.csv")              # 예측 대상 쌍(question1,question2)
print("pos pairs", len(train), "| raw", len(raw), "| test", len(test))

In [ ]:
import random, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
random.seed(0); torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
MP = "data/my_custom_bert_model"
tok = AutoTokenizer.from_pretrained(MP)
model = AutoModelForSequenceClassification.from_pretrained(MP, num_labels=2).to(dev)

# 학습쌍: 양성(400) + raw 조합 음성(400)
q1 = list(train.question1); q2 = list(train.question2); y = [1]*len(train)
rq = list(raw.question); random.shuffle(rq)
for i in range(0, 2*len(train), 2):
    q1.append(rq[i % len(rq)]); q2.append(rq[(i+1) % len(rq)]); y.append(0)

def encode(a, b):
    e = tok(list(a), list(b), truncation=True, max_length=64, padding="max_length", return_tensors="pt")
    return e["input_ids"], e["attention_mask"]
ids, mask = encode(q1, q2)
loader = DataLoader(TensorDataset(ids, mask, torch.tensor(y)), batch_size=16, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), 2e-5)
model.train()
for ep in range(3):
    for bi, bm, by in loader:
        opt.zero_grad()
        out = model(input_ids=bi.to(dev), attention_mask=bm.to(dev), labels=by.to(dev))
        out.loss.backward(); opt.step()
    print("epoch", ep, "loss", round(float(out.loss), 4))

model.eval(); preds = []
tids, tmask = encode(test.question1, test.question2)
with torch.no_grad():
    for i in range(0, len(tids), 64):
        logits = model(input_ids=tids[i:i+64].to(dev), attention_mask=tmask[i:i+64].to(dev)).logits
        preds += logits.argmax(1).cpu().tolist()

predictions = {str(i): int(preds[i]) for i in range(len(test))}
with open("submission.json", "w") as f:
    json.dump(predictions, f, indent=4)
print("saved submission.json", len(predictions), "| pos", sum(preds))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)